In [3]:
import os
from pathlib import Path
ROOT = "../" # Base directory relative to this notebook's location. Adjust if the notebook is moved
CSV_DIR = os.path.join(ROOT, "csv", "historical-data")

import pandas as pd

In [11]:
def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def safe_stat(df, column, func):
    if column not in df.columns:
        return None

    return func(safe_numeric(df[column]))


rows = []

processed = 0

for filename in os.listdir(CSV_DIR):

    file_path = os.path.join(CSV_DIR, filename)

    # Ignore subfolders
    if not os.path.isfile(file_path):
        continue

    try:
        # Force indicativo as text
        df = pd.read_csv(file_path, dtype={"indicativo": str})

        print(f"Processing {filename}... ({processed + 1}/{len(os.listdir(CSV_DIR))})")

        rows.append(
            {
                "indicativo": str(df["indicativo"].iloc[0]) if "indicativo" in df.columns else None,
                "nombre": df["nombre"].iloc[0] if "nombre" in df.columns else None,
                "altitud": df["altitud"].iloc[0] if "altitud" in df.columns else None,
                "provincia": df["provincia"].iloc[0] if "provincia" in df.columns else None,
                "num_records": len(df),
                "avg_tmin": safe_stat(df, "tmin", lambda s: s.mean()),
                "avg_tmax": safe_stat(df, "tmax", lambda s: s.mean()),
                "avg_prec": safe_stat(df, "prec", lambda s: s.mean()),
                "avg_velmedia": safe_stat(df, "velmedia", lambda s: s.mean()),
                "std_tmed": safe_stat(df, "tmed", lambda s: s.std()),
            }
        )

        processed += 1

    except Exception as e:
        print(f"Error processing {filename}: {e}")

stations_df = pd.DataFrame(rows)

# Ensure consistent type before sorting
stations_df["indicativo"] = stations_df["indicativo"].astype(str)

stations_df = (
    stations_df
    .sort_values("indicativo")
    .reset_index(drop=True)
)

display(stations_df.head())

Processing 0009X_hist.csv... (1/770)
Processing 0016A_hist.csv... (2/770)
Processing 0034X_hist.csv... (3/770)
Processing 0042Y_hist.csv... (4/770)
Processing 0061X_hist.csv... (5/770)
Processing 0066X_hist.csv... (6/770)
Processing 0073X_hist.csv... (7/770)
Processing 0076_hist.csv... (8/770)
Processing 0092X_hist.csv... (9/770)
Processing 0106X_hist.csv... (10/770)
Processing 0114X_hist.csv... (11/770)
Processing 0120X_hist.csv... (12/770)
Processing 0149X_hist.csv... (13/770)
Processing 0158X_hist.csv... (14/770)
Processing 0171X_hist.csv... (15/770)
Processing 0194D_hist.csv... (16/770)
Processing 0201X_hist.csv... (17/770)
Processing 0222X_hist.csv... (18/770)
Processing 0244X_hist.csv... (19/770)
Processing 0260X_hist.csv... (20/770)
Processing 0281Y_hist.csv... (21/770)
Processing 0284X_hist.csv... (22/770)
Processing 0312X_hist.csv... (23/770)
Processing 0320I_hist.csv... (24/770)
Processing 0360X_hist.csv... (25/770)
Processing 0363X_hist.csv... (26/770)
Processing 0367_hist.c

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
0,0009X,ALFORJA,406,TARRAGONA,1760,10.326706,21.866588,1.275278,2.997434,6.333601
1,0016A,REUS AEROPUERTO,71,TARRAGONA,1950,11.727940,22.801129,1.312099,3.569625,6.325506
2,0034X,VALLS,233,TARRAGONA,1950,10.850077,22.453155,1.128718,NaN,6.341960
3,0042Y,TARRAGONA,55,TARRAGONA,1934,13.167755,22.431363,1.341408,NaN,5.705496
4,0061X,PONTONS,632,BARCELONA,1911,8.404505,19.591200,1.534488,3.449710,6.158788


In [13]:
stations_df.to_csv(os.path.join(ROOT, "csv", "stats", "weather-station-stats.csv"), index=False)

In [54]:
# Highest Average High Temperature
highest_tmax = stations_df.sort_values("avg_tmax", ascending=False).head(10)
display(
    highest_tmax.style
    .set_properties(subset=["avg_tmax"], **{"background-color": "#ff9999", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
386,5514Z,GRANADA BASE AÉREA,695,GRANADA,1485,11.022811,26.961899,0.723028,3.324672,7.398497
742,C628B,"LA ALDEA DE SAN NICOLÁS, TASARTE",318,LAS PALMAS,1950,17.192204,26.949561,0.331298,3.919077,3.911497
406,5790Y,"SEVILLA, TABLADA",9,SEVILLA,1620,13.545730,26.834839,1.306117,1.843272,6.332577
399,5702X,CARMONA,50,SEVILLA,1781,12.687255,26.799712,1.398119,2.732022,6.687588
375,5361X,MONTORO,155,CORDOBA,1916,10.789443,26.653519,1.456227,1.493058,7.436028
396,5641X,ÉCIJA,130,SEVILLA,1939,12.402801,26.494767,1.315734,2.182128,7.020379
716,C319W,"VALLEHERMOSO, DAMA",190,STA. CRUZ DE TENERIFE,1951,16.812199,26.478934,0.368016,2.545231,2.768967
349,4541X,EL GRANADO,60,HUELVA,1959,12.172344,26.257812,1.220459,nan,6.379955
407,5796,MORÓN DE LA FRONTERA,87,SEVILLA,1951,12.719795,26.241077,1.491203,2.035457,6.714411
476,7178I,MURCIA,62,MURCIA,1951,14.048642,26.176269,0.924541,2.570771,6.576313


In [55]:
# Lowest Average High Temperature
lowest_tmax = stations_df.sort_values("avg_tmax", ascending=False).tail(10)
display(lowest_tmax.style.set_properties(subset=["avg_tmax"], **{"background-color": "#829cd4", "color": "black"}))

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
593,9445L,"FORMIGAL, SARRIOS",1800,HUESCA,1803,4.086832,11.246936,4.802456,nan,6.711358
537,9001S,ALTO CAMPOO,1650,CANTABRIA,1856,3.685179,11.160055,3.183259,3.203261,6.045328
594,9451F,"PANTICOSA, PETROSOS",1850,HUESCA,1945,4.417929,11.109381,3.646262,3.125750,6.794204
87,1221D,PAJARES-VALGRANDE,1480,ASTURIAS,1893,3.924867,11.072476,4.410551,2.953196,5.867952
634,9814I,"TORLA-ORDESA, EL CEBOLLAR",1905,HUESCA,1950,3.772858,10.998358,3.387566,2.919276,6.671582
72,1167G,"MIRADOR DEL CABLE, PARQUE NACIONAL PICOS DE EUROPA",1910,CANTABRIA,373,2.395148,8.139892,2.676829,5.366860,6.023616
636,9839V,"CERLER, COGULLA",2374,HUESCA,1956,1.318661,7.749233,2.317997,3.623402,6.594720
73,1167J,"CORISCAO, PARQUE NACIONAL PICOS DE EUROPA",1722,CANTABRIA,330,2.232508,7.576780,2.593403,3.265484,5.415132
653,9988B,CAP DE VAQUÈIRA,2467,LLEIDA,1928,0.160863,6.258733,nan,4.183624,6.921823
618,9677,PORT AINÉ,2410,LLEIDA,1887,0.372395,6.095489,nan,5.185623,6.936063


In [56]:
# Highest Average Low Temperature
highest_tmin = stations_df.sort_values("avg_tmin", ascending=False).head(10)

display(
    highest_tmin.style
    .set_properties(subset=["avg_tmin"], **{"background-color": "#f07d12", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
718,C329Z,SAN SEBASTIÁN DE LA GOMERA,15,STA. CRUZ DE TENERIFE,1260,19.898756,25.385841,0.303095,3.372816,2.962153
766,C929I,HIERRO AEROPUERTO,32,SANTA CRUZ DE TENERIFE,1956,19.710838,23.813139,0.420661,6.555003,2.306746
708,C229J,PÁJARA,15,LAS PALMAS,1939,19.645003,25.237253,0.199895,3.251475,2.926524
755,C659M,"LAS PALMAS DE GRAN CANARIA, PL. DE LA FERIA",15,LAS PALMAS,1950,19.583297,23.715343,0.399169,2.014308,2.339005
729,C449C,STA.CRUZ DE TENERIFE,36,SANTA CRUZ DE TENERIFE,1800,19.560468,25.722871,0.517420,3.016215,2.863172
743,C629Q,"MOGÁN, PUERTO RICO",10,LAS PALMAS,1959,19.519642,23.754936,0.127535,3.023430,2.346820
738,C619X,AGAETE,5,LAS PALMAS,1956,19.405541,24.163725,0.281795,4.824591,2.656844
705,C129V,FUENCALIENTE,19,STA. CRUZ DE TENERIFE,1953,19.305598,24.736775,0.514710,6.374165,2.366184
767,C939T,"FRONTERA, SABINOSA",20,STA. CRUZ DE TENERIFE,1960,19.080490,23.663483,0.467776,3.213950,2.417431
746,C639M,"MASPALOMAS, C. INSULAR TURISMO",45,LAS PALMAS,1954,18.881048,25.762661,0.215670,2.752968,2.974474


In [57]:
# Lowest Average Low Temperature
lowest_tmin = stations_df.sort_values("avg_tmin", ascending=False).tail(10)
display(lowest_tmin.style.set_properties(subset=["avg_tmin"], **{"background-color": "#cacbf1", "color": "black"}))

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
617,9657X,ESTERRI D'ÀNEU,952,LLEIDA,1928,3.237532,17.436886,1.649341,1.346782,6.287631
212,2766E,"SANABRIA, ROBLEDA-CERVANTES",933,ZAMORA,1956,2.780974,18.100000,2.580419,0.973900,5.878979
203,2630X,PUERTO DE SAN ISIDRO,1510,LEON,1664,2.471739,12.333737,2.682786,3.876034,5.897696
72,1167G,"MIRADOR DEL CABLE, PARQUE NACIONAL PICOS DE EUROPA",1910,CANTABRIA,373,2.395148,8.139892,2.676829,5.366860,6.023616
73,1167J,"CORISCAO, PARQUE NACIONAL PICOS DE EUROPA",1722,CANTABRIA,330,2.232508,7.576780,2.593403,3.265484,5.415132
612,9590D,CAP DE REC,1940,LLEIDA,1912,2.111432,11.359150,nan,1.536597,6.359448
264,3319D,PUERTO DEL PICO,1285,AVILA,1957,1.669646,16.723267,4.546848,2.477466,6.078949
636,9839V,"CERLER, COGULLA",2374,HUESCA,1956,1.318661,7.749233,2.317997,3.623402,6.594720
618,9677,PORT AINÉ,2410,LLEIDA,1887,0.372395,6.095489,nan,5.185623,6.936063
653,9988B,CAP DE VAQUÈIRA,2467,LLEIDA,1928,0.160863,6.258733,nan,4.183624,6.921823


In [44]:
# Highest Average Precipitation
highest_prec = stations_df.sort_values("avg_prec", ascending=False).head(10)
display(
    highest_prec.style
    .set_properties(subset=["avg_prec"], **{"background-color": "#a1cfff", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
123,1476R,"ROIS, CASAS DO PORTO",210,A CORUÑA,1937,10.137051,19.227581,8.967377,nan,4.726292
113,1406X,MAZARICOS,340,A CORUÑA,1944,9.491486,17.618998,8.650723,nan,4.539642
126,1489A,A LAMA,395,PONTEVEDRA,1944,8.441434,18.750310,7.388579,2.261368,5.188006
143,1696O,BEARIZ,610,OURENSE,1940,5.228883,19.492100,6.726098,nan,5.361435
120,1468X,A ESTRADA,269,PONTEVEDRA,1944,9.212519,19.643998,6.554127,nan,5.103344
413,5911A,GRAZALEMA,913,CADIZ,1924,10.117900,20.344678,6.444491,1.549738,6.548668
111,1399,VIMIANZO,287,A CORUÑA,1948,9.405095,17.838909,6.249275,nan,4.252636
127,1495,VIGO AEROPUERTO,255,PONTEVEDRA,1951,10.356540,19.367267,6.098723,3.495003,4.966618
147,1719,A CAÑIZA,560,PONTEVEDRA,1955,8.788775,18.312711,5.718272,nan,5.244065
39,1021X,"ERRENTERIA, AÑARBE",165,GIPUZKOA,1940,10.125838,19.949149,5.656669,1.556605,5.404259


In [43]:
# Lowest Avg Precipitation
lowest_prec = (
    stations_df.dropna(subset=["avg_prec"])
    .sort_values("avg_prec", ascending=True)
    .head(10)
)

display(
    lowest_prec.style
    .set_properties(subset=["avg_prec"], **{"background-color": "#e2d40e", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
743,C629Q,"MOGÁN, PUERTO RICO",10,LAS PALMAS,1959,19.519642,23.754936,0.127535,3.023430,2.346820
759,C689E,MASPALOMAS,6,LAS PALMAS,1955,18.375424,24.325989,0.160733,3.378699,2.805218
709,C239N,"TUINEJE,PUERTO GRAN TARAJAL",1,LAS PALMAS,1960,18.240245,25.273289,0.176982,4.050051,3.322895
699,C019V,YAIZA PLAYA BLANCA,6,LAS PALMAS,1947,18.362204,24.161946,0.187197,4.139548,2.803016
744,C629X,"MOGÁN, PUERTO",10,LAS PALMAS,1940,18.470775,25.358866,0.187250,3.068711,2.640014
711,C249I,FUERTEVENTURA AEROPUERTO,25,LAS PALMAS,1960,18.394493,24.543284,0.195838,6.168615,2.825682
708,C229J,PÁJARA,15,LAS PALMAS,1939,19.645003,25.237253,0.199895,3.251475,2.926524
746,C639M,"MASPALOMAS, C. INSULAR TURISMO",45,LAS PALMAS,1954,18.881048,25.762661,0.215670,2.752968,2.974474
760,C839X,LA GRACIOSA,19,LAS PALMAS,1872,18.158897,24.280513,0.241173,5.864115,2.624452
763,C919K,TACORON-LAPILLAS-TORTUGA,98,STA. CRUZ DE TENERIFE,1753,18.868021,25.066705,0.241991,3.334914,2.629190


In [46]:
# Highest average wind speed
highest_wind = stations_df.sort_values("avg_velmedia", ascending=False).head(10)
display(
    highest_wind.style
    .set_properties(subset=["avg_velmedia"], **{"background-color": "#ff99f7", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
104,1351,ESTACA DE BARES,90,A CORUÑA,1946,12.567962,16.859034,2.217276,8.473584,3.395864
750,C649I,GRAN CANARIA AEROPUERTO,24,LAS PALMAS,1951,18.771754,25.044568,0.391115,8.145473,2.797182
713,C314Z,"VALLEHERMOSO, ALTO IGUALERO",1474,STA. CRUZ DE TENERIFE,1934,10.436115,18.011694,1.470384,6.957210,6.102387
724,C430E,IZAÑA,2369,SANTA CRUZ DE TENERIFE,1440,7.467014,15.419444,0.557344,6.809843,5.900317
110,1393,CABO VILÁN,50,A CORUÑA,1922,11.957752,16.971391,3.387513,6.787197,3.187627
112,1400,FISTERRA,230,A CORUÑA,1950,11.846570,17.011449,2.968671,6.779914,3.687926
766,C929I,HIERRO AEROPUERTO,32,SANTA CRUZ DE TENERIFE,1956,19.710838,23.813139,0.420661,6.555003,2.306746
705,C129V,FUENCALIENTE,19,STA. CRUZ DE TENERIFE,1953,19.305598,24.736775,0.514710,6.374165,2.366184
723,C429I,TENERIFE SUR AEROPUERTO,64,SANTA CRUZ DE TENERIFE,1956,18.187538,26.027408,0.322690,6.256511,3.006532
711,C249I,FUERTEVENTURA AEROPUERTO,25,LAS PALMAS,1960,18.394493,24.543284,0.195838,6.168615,2.825682


In [47]:
# Lowest average wind speed
lowest_wind = (
    stations_df.dropna(subset=["avg_velmedia"])
    .sort_values("avg_velmedia", ascending=True)
    .head(10)
)
display(
    lowest_wind.style
    .set_properties(subset=["avg_velmedia"], **{"background-color": "#99ff99", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
427,6040X,CORTES DE LA FRONTERA,315,MALAGA,1919,11.926600,23.561438,2.590456,0.709183,6.015378
24,0360X,LES PLANES D'HOSTOLES,337,GIRONA,1931,6.727268,21.500209,2.104973,0.759206,6.567517
144,1700X,O CARBALLIÑO,400,OURENSE,1942,7.450000,20.503021,3.445534,0.785044,5.895863
661,B103B,ANDRATX - SANT ELM,52,BALEARES,1933,13.120084,22.835310,1.069375,0.855898,5.934885
595,9453X,"BIESCAS, EMBALSE DE BÚBAL",1100,HUESCA,1960,4.672699,16.557077,4.111225,0.870787,6.494747
388,5515X,GRANADA-CARTUJA,775,GRANADA,1940,10.954474,24.829723,0.902431,0.903026,7.556050
274,3423I,MADRIGAL DE LA VERA,464,CACERES,1960,11.186821,22.931436,3.644410,0.906605,7.409729
658,B013X,"ESCORCA, LLUC",490,ILLES BALEARS,1951,9.725748,21.192879,2.819458,0.938779,6.406004
637,9843A,SEIRA,825,HUESCA,1956,5.840153,19.437769,2.951509,0.953476,6.965728
212,2766E,"SANABRIA, ROBLEDA-CERVANTES",933,ZAMORA,1956,2.780974,18.100000,2.580419,0.973900,5.878979


In [49]:
# Highest Temperature Variability (std of tmed)
highest_variability = stations_df.sort_values("std_tmed", ascending=False).head(10)
display(
    highest_variability.style
    .set_properties(subset=["std_tmed"], **{"background-color": "#c80dda", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
300,4064Y,ALCAZAR DE SAN JUAN,640,CIUDAD REAL,1943,10.613967,23.016917,0.901867,1.623569,8.333520
616,9650X,ARTESA DE SEGRE,400,LLEIDA,1951,7.652464,21.474692,1.091812,nan,8.076644
314,4147X,VALDEPEÑAS,700,CIUDAD REAL,1944,10.027324,22.771864,0.895273,1.815396,8.068184
239,3085Y,PASTRANA,920,GUADALAJARA,1829,7.858475,20.707627,1.536581,nan,8.062121
315,4148,VISO DEL MARQUÉS,804,CIUDAD REAL,1871,9.381059,22.340128,1.216122,2.742460,8.039264
312,4121,CIUDAD REAL,626,CIUDAD REAL,1960,10.560245,22.927477,1.153240,2.104972,8.037380
360,5038Y,CAZORLA,799,JAEN,1952,12.634857,23.633455,1.548975,nan,8.034260
310,4103X,TOMELLOSO,662,CIUDAD REAL,1894,9.245445,23.196590,0.908445,2.774054,8.026180
311,4116I,ALMAGRO / FAMET,626,CIUDAD REAL,1086,8.553598,23.378967,0.847304,2.950877,8.021111
621,9707,LLIMIANA,515,LLEIDA,1910,7.441777,22.236277,1.533828,1.816632,8.002564


In [53]:
# Lowest Temperature Variability (std of tmed)
lowest_variability = (
    stations_df.dropna(subset=["std_tmed"])
    .sort_values("std_tmed", ascending=True)
    .head(10)
)
display(
    lowest_variability.style.set_properties(subset=["std_tmed"], **{"background-color": "#cbb0cc", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed
766,C929I,HIERRO AEROPUERTO,32,SANTA CRUZ DE TENERIFE,1956,19.710838,23.813139,0.420661,6.555003,2.306746
755,C659M,"LAS PALMAS DE GRAN CANARIA, PL. DE LA FERIA",15,LAS PALMAS,1950,19.583297,23.715343,0.399169,2.014308,2.339005
754,C659H,"LAS PALMAS DE GRAN CANARIA, SAN CRISTOBAL",55,LAS PALMAS,1934,18.397410,23.692364,0.356768,3.954835,2.344277
743,C629Q,"MOGÁN, PUERTO RICO",10,LAS PALMAS,1959,19.519642,23.754936,0.127535,3.023430,2.346820
705,C129V,FUENCALIENTE,19,STA. CRUZ DE TENERIFE,1953,19.305598,24.736775,0.514710,6.374165,2.366184
767,C939T,"FRONTERA, SABINOSA",20,STA. CRUZ DE TENERIFE,1960,19.080490,23.663483,0.467776,3.213950,2.417431
706,C139E,LA PALMA AEROPUERTO,33,SANTA CRUZ DE TENERIFE,1956,18.749846,23.493859,0.812104,5.336280,2.423936
758,C669B,ARUCAS,86,LAS PALMAS,1951,17.926756,23.583006,0.450880,2.531709,2.464945
733,C459Z,PUERTO DE LA CRUZ,25,STA. CRUZ DE TENERIFE,1955,18.490685,24.706794,0.756909,2.673468,2.482047
760,C839X,LA GRACIOSA,19,LAS PALMAS,1872,18.158897,24.280513,0.241173,5.864115,2.624452
